# Modernized IMDB + DeBERTa + P-Tuning

In [1]:
%pip install -q --upgrade pip
%pip install -q "transformers>=5.14,<6" "peft>=0.20,<0.21" datasets evaluate accelerate scikit-learn pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

import sys
import math
import logging
import random

import numpy as np
import pandas as pd
import torch
import datasets
import evaluate

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import PromptEncoderConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split


In [3]:
# === Configuration (edit me) ===
MODEL_ID = "microsoft/deberta-v3-small"
TRAIN_PATH = "/kaggle/input/datasets/kintsugi0v0/imdb-data/labeledTrainData.tsv"
TEST_PATH = "/kaggle/input/datasets/kintsugi0v0/imdb-data/testData.tsv"
OUTPUT_DIR = "./checkpoint"
RESULT_DIR = "./result"
RESULT_FILENAME = "deberta_ptuning.csv"
RESULT_PATH = f"{RESULT_DIR}/{RESULT_FILENAME}"
NUM_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 4
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 100
SEED = 42
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
NUM_VIRTUAL_TOKENS = 20
ENCODER_HIDDEN_SIZE = 128


In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)


In [5]:
program = os.path.basename(sys.argv[0]) if sys.argv else "notebook"
logger = logging.getLogger(program)
logging.basicConfig(format="%(asctime)s: %(levelname)s: %(message)s")
logging.root.setLevel(level=logging.INFO)
logger.info("running %s", program)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)


2026-08-05 05:30:08,669: INFO: running colab_kernel_launcher.py


In [6]:
train_df = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
test_df = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=SEED)

train_dataset = datasets.Dataset.from_dict({"label": train_df["sentiment"], "text": train_df["review"]})
val_dataset = datasets.Dataset.from_dict({"label": val_df["sentiment"], "text": val_df["review"]})
test_dataset = datasets.Dataset.from_dict({"text": test_df["review"]})


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)


tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


2026-08-05 05:30:10,870: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 05:30:10,871: WARNING: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-05 05:30:10,880: INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/config.json "HTTP/1.1 200 OK"
2026-08-05 05:30:10,889: INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

2026-08-05 05:30:11,118: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 05:30:11,126: INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-05 05:30:11,135: INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

2026-08-05 05:30:11,347: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-05 05:30:11,550: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-05 05:30:11,752: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/spm.model "HTTP/1.1 302 Found"
2026-08-05 05:30:11,977: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/xet-read-token/a36c739020e01763fe789b4b85e2df55d6180012 "HTTP/1.1 200 OK"


spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

2026-08-05 05:30:14,100: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
2026-08-05 05:30:14,295: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-08-05 05:30:14,499: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
2026-08-05 05:30:14,698: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-08-05 05:30:15,319: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small "HTTP/1.1 200 OK"


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
)


2026-08-05 05:30:39,583: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 05:30:39,590: INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/config.json "HTTP/1.1 200 OK"
2026-08-05 05:30:39,791: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-08-05 05:30:40,024: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 05:30:40,032: INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/config.json "HTTP/1.1 200 OK"
2026-08-05 05:30:40,232: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/model.safetensors "

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

2026-08-05 05:30:46,474: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-08-05 05:30:46,670: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifie

In [9]:
peft_config = PromptEncoderConfig(
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    encoder_hidden_size=ENCODER_HIDDEN_SIZE,
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


trainable params: 230,914 || all params: 142,127,364 || trainable%: 0.1625


In [10]:
metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


2026-08-05 05:30:46,938: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/commits/main "HTTP/1.1 200 OK"
2026-08-05 05:30:47,157: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/discussions?p=0 "HTTP/1.1 200 OK"
2026-08-05 05:30:47,369: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/commits/refs%2Fpr%2F4 "HTTP/1.1 200 OK"
2026-08-05 05:30:47,569: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/refs%2Fpr%2F4/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-08-05 05:30:47,772: INFO: HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/refs%2Fpr%2F4/model.safetensors "HTTP/1.1 302 Found"
2026-08-05 05:30:47,972: INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/xet-read-token/a59be8aa63396e73dbb45a1487e4cde4be98bfa4 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [11]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    eval_strategy="epoch",
)


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [13]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.910491,1.149491,0.806000
2,1.221194,0.953331,0.830800
3,1.026143,1.036588,0.831400


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=15000, training_loss=1.0388204762776694, metrics={'train_runtime': 2444.745, 'train_samples_per_second': 24.542, 'train_steps_per_second': 6.136, 'total_flos': 8591461639594656.0, 'train_loss': 1.0388204762776694, 'epoch': 3.0})

In [14]:
prediction_outputs = trainer.predict(tokenized_test)
test_pred = np.argmax(prediction_outputs[0], axis=-1).flatten()
print(test_pred[:10])

result_output = pd.DataFrame(data={"id": test_df["id"], "sentiment": test_pred})
result_output.to_csv(RESULT_PATH, index=False, quoting=3)
logger.info("result saved to %s", RESULT_PATH)


2026-08-05 06:17:35,075: INFO: result saved to ./result/deberta_ptuning.csv


[1 0 1 1 1 1 0 1 0 1]
